# imports

In [3]:
import re

from babel.dates import match_skeleton

# Dataset

In [1]:
SQL = {
    "page1": """
آکادمی فناوری آینده | راهنمای بوتکمپ هوش مصنوعی
نسخه سند: ۱۴۰۵/۰۲

بخش اول: قوانین حضور و غیاب

حضور در كلاس‌های آنلاین الزامی است.
دانشجويان موظف‌اند حداقل در ۸۰ درصد جلسات شرکت کنند.

در صورت غیبت بیش از ۳ جلسه، وضعیت ادامه حضور
دانشجو توسط واحد آموزش بررسی خواهد شد.

برای ثبت درخواست غیبت موجه، دانشجو باید
حداکثر تا ۴۸ ساعت پس از جلسه، درخواست خود را
در سامانه آموزشی ثبت کند.

آکادمی فناوری آینده | پشتیبانی: support@futureacademy.ir
صفحه ۱ از ۳
""",
    "page2": """
آکادمی فناوری آینده | راهنمای بوتکمپ هوش مصنوعی
نسخه سند: ۱۴۰۵/۰۲

بخش دوم: تکالیف و ارزیابی

تكاليف هر هفته باید تا ساعت ۲۳:۵۹ روز جمعه
در سامانه آموزشی ثبت شوند.

ارسال تكاليف پس از مهلت مقرر ممکن است
باعث کاهش امتیاز ارزیابی شود.

برای مشکلات مربوط به بارگذاری تکالیف،
دانشجویان می‌توانند با نشانی
assignments@futureacademy.ir مکاتبه کنند.

توجه: شماره صفحه اعلام‌شده در فرم ارزیابی
باید با شماره صفحه گزارش ارسالی مطابقت داشته باشد.

آکادمی فناوری آینده | پشتیبانی: support@futureacademy.ir
صفحه ۲ از ۳
""",
    "page3": """
آکادمی فناوری آینده | راهنمای بوتکمپ هوش مصنوعی
نسخه سند: ۱۴۰۵/۰۲

بخش سوم: گواهی پایان دوره

دریافت گواهی پایان دوره منوط به تکمیل
حداقل ۸۰ درصد تکالیف و شرکت در ارزیابی نهایی است.

در گواهی صادرشده، عنوان رسمی دوره
«بوتکمپ هوش مصنوعی آکادمی فناوری آینده»
درج خواهد شد.

دانشجویانی که شرایط دریافت گواهی را دارند،
می‌توانند وضعیت صدور آن را از طریق
certificates@futureacademy.ir پیگیری کنند.

آکادمی فناوری آینده | پشتیبانی: support@futureacademy.ir
صفحه ۳ از ۳
"""
}

# Cleaner_function

In [16]:
def is_header_or_footer(line):
    """بررسی اینکه آیا خط جزو هدر یا فوترهای روتین و تکراری هست یا نه."""
    line = line.strip()
    if not line:
        return False

    # الگوهای هدر
    if "آکادمی فناوری آینده | راهنمای بوتکمپ" in line:
        return True
    if line.startswith("نسخه سند:"):
        return True

    # الگوهای فوتر
    if "پشتیبانی: support@futureacademy.ir" in line:
        return True
    # شماره صفحه: مثلا 'صفحه ۱ از ۳' یا 'صفحه 2 از 3'
    if re.search(r"صفحه\s+[\d\u06F0-\u06F9]+\s+از\s+[\d\u06F0-\u06F9]+", line):
        return True

    return False

def cleaner_rag(dataset=SQL):
    """This funciton will retrun 4 keys: pagenumber, raw text , clean text, removed lines"""
    results = []

    for page_key, raw_text in dataset.items():
        lines = raw_text.splitlines()

        clean_lines = []
        removed_lines = []

        for line in lines:
            line_stripped = line.strip()

            # خطوط خالی رو نادیده می‌گیریم تا متن تمیزتر بشه
            if not line_stripped:
                continue

            # اگر هدر یا فوتر بود، می‌فرستیم توی حذفیات
            if is_header_or_footer(line_stripped):
                removed_lines.append(line_stripped)
            else:
                clean_lines.append(line_stripped)

        page_num_match = re.search(r"\d+", page_key)
        page_number = int(page_num_match.group()) if page_num_match else page_key

        results.append(
            {
                "pagenumber": page_number,
                "raw_text": raw_text.strip(),
                "clean_text": "\n".join(clean_lines),
                "removed_lines": removed_lines,
            }
        )

    return results


sql_results = cleaner_rag(SQL)

print(f"keys : {sql_results[0].keys()}")
print(f"clean text example:\n{sql_results[2]['clean_text']}\n{"-"*10}\n")
print(f"raw text example:\n{sql_results[0]['raw_text']}")
print(f"")


keys : dict_keys(['pagenumber', 'raw_text', 'clean_text', 'removed_lines'])
clean text example:
بخش سوم: گواهی پایان دوره
دریافت گواهی پایان دوره منوط به تکمیل
حداقل ۸۰ درصد تکالیف و شرکت در ارزیابی نهایی است.
در گواهی صادرشده، عنوان رسمی دوره
«بوتکمپ هوش مصنوعی آکادمی فناوری آینده»
درج خواهد شد.
دانشجویانی که شرایط دریافت گواهی را دارند،
می‌توانند وضعیت صدور آن را از طریق
certificates@futureacademy.ir پیگیری کنند.
----------

raw text example:
آکادمی فناوری آینده | راهنمای بوتکمپ هوش مصنوعی
نسخه سند: ۱۴۰۵/۰۲

بخش اول: قوانین حضور و غیاب

حضور در كلاس‌های آنلاین الزامی است.
دانشجويان موظف‌اند حداقل در ۸۰ درصد جلسات شرکت کنند.

در صورت غیبت بیش از ۳ جلسه، وضعیت ادامه حضور
دانشجو توسط واحد آموزش بررسی خواهد شد.

برای ثبت درخواست غیبت موجه، دانشجو باید
حداکثر تا ۴۸ ساعت پس از جلسه، درخواست خود را
در سامانه آموزشی ثبت کند.

آکادمی فناوری آینده | پشتیبانی: support@futureacademy.ir
صفحه ۱ از ۳



# تمرین 2

In [17]:
text = """
بند ۱: حضور در کلاس‌های آنلاین الزامی است.
بند ۲: دانشجویان باید حداقل در ۸۰ درصد جلسات حضور داشته باشند.
بند ۳: غیبت بیش از سه جلسه نیازمند بررسی واحد آموزش است.
بند ۴: درخواست غیبت موجه باید حداکثر تا ۴۸ ساعت پس از جلسه ثبت شود.
بند ۵: تکالیف هفتگی تا ساعت ۲۳:۵۹ جمعه دریافت می‌شوند.
بند ۶: دانشجو باید پیش از پایان مهلت از ثبت موفق فایل اطمینان حاصل کند.
بند ۷: اگر دانشجو به دلیل اختلال سامانه نتواند تکلیف را در مهلت مقرر ارسال کند، موظف است حداکثر تا ۲۴ ساعت پس از پایان مهلت تصویر خطای سامانه و مشخصات تکلیف را به پشتیبانی بفرستد تا درخواست او بررسی شود.
بند ۸: دریافت گواهی به تکمیل ۸۰ درصد تکالیف و شرکت در ارزیابی نهایی وابسته است.
"""

In [43]:
pattern = r"(?m)^[ \t]*بند[ \t]*[۰-۹٠-٩0-9]+[ \t]*:"
matches = list(re.finditer(pattern, text))

clauses = []
for i, match in enumerate(matches):
    start = match.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

    # فاصله‌ها و خط‌های جدید را یکدست می‌کنیم.
    clause = re.sub(r"\s+", " ", text[start:end]).strip()
    clauses.append(clause)

clauses

['بند ۱: حضور در کلاس\u200cهای آنلاین الزامی است.',
 'بند ۲: دانشجویان باید حداقل در ۸۰ درصد جلسات حضور داشته باشند.',
 'بند ۳: غیبت بیش از سه جلسه نیازمند بررسی واحد آموزش است.',
 'بند ۴: درخواست غیبت موجه باید حداکثر تا ۴۸ ساعت پس از جلسه ثبت شود.',
 'بند ۵: تکالیف هفتگی تا ساعت ۲۳:۵۹ جمعه دریافت می\u200cشوند.',
 'بند ۶: دانشجو باید پیش از پایان مهلت از ثبت موفق فایل اطمینان حاصل کند.',
 'بند ۷: اگر دانشجو به دلیل اختلال سامانه نتواند تکلیف را در مهلت مقرر ارسال کند، موظف است حداکثر تا ۲۴ ساعت پس از پایان مهلت تصویر خطای سامانه و مشخصات تکلیف را به پشتیبانی بفرستد تا درخواست او بررسی شود.',
 'بند ۸: دریافت گواهی به تکمیل ۸۰ درصد تکالیف و شرکت در ارزیابی نهایی وابسته است.']

In [44]:

def chunk_by_bands(text, max_words=25):
    """
    متن را بر اساس بندها چانک می‌کند.
    - برچسب «بند n:» جزو تعداد کلمات است.
    - هیچ بندی شکسته نمی‌شود.
    - اگر خود یک بند از سقف بلندتر باشد، به‌تنهایی یک چانک می‌شود.
    """

    if max_words <= 0:
        raise ValueError("max_words باید عددی بزرگ‌تر از صفر باشد.")

    # ابتدای هر بند را پیدا می‌کنیم؛ شماره می‌تواند فارسی، عربی یا انگلیسی باشد.
    pattern = r"(?m)^[ \t]*بند[ \t]*[۰-۹٠-٩0-9]+[ \t]*:"
    matches = list(re.finditer(pattern, text))

    if not matches:
        raise ValueError("هیچ بندی با قالب «بند n:» پیدا نشد.")

    # اگر قبل از بند اول متن اضافه‌ای وجود داشته باشد، خطا می‌دهیم
    # تا متن ناخواسته بی‌صدا حذف نشود.
    if text[:matches[0].start()].strip():
        raise ValueError("قبل از بند اول، متنِ خارج از قالب بند وجود دارد.")

    # متن را به بندهای کامل تقسیم می‌کنیم.
    clauses = []
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        # فاصله‌ها و خط‌های جدید را یکدست می‌کنیم.
        clause = re.sub(r"\s+", " ", text[start:end]).strip()
        clauses.append(clause)

    chunks = []
    current_clauses = []
    current_word_count = 0

    for clause in clauses:
        clause_word_count = len(clause.split())

        # اگر بند کامل جا نشود، چانک فعلی را می‌بندیم.
        # خود بند را نصف نمی‌کنیم.
        if current_clauses and current_word_count + clause_word_count > max_words:
            chunks.append({
                "text": "\n".join(current_clauses),
                "word_count": current_word_count,
                "over_limit": current_word_count > max_words,
            })
            current_clauses = []
            current_word_count = 0

        # بند کامل به چانک اضافه می‌شود؛ حتی اگر به‌تنهایی از سقف بلندتر باشد.
        current_clauses.append(clause)
        current_word_count += clause_word_count

    # ثبت آخرین چانک
    if current_clauses:
        chunks.append({
            "text": "\n".join(current_clauses),
            "word_count": current_word_count,
            "over_limit": current_word_count > max_words,
        })

    return chunks


In [45]:
chunks = chunk_by_bands(text, max_words=25)

for i, chunk in enumerate(chunks, start=1):
    print(
        f"چانک {i} | "
        f"{chunk['word_count']} کلمه | "
        f"بیشتر از سقف: {chunk['over_limit']}"
    )
    print(chunk["text"])
    print("-" * 40)


چانک 1 | 20 کلمه | بیشتر از سقف: False
بند ۱: حضور در کلاس‌های آنلاین الزامی است.
بند ۲: دانشجویان باید حداقل در ۸۰ درصد جلسات حضور داشته باشند.
----------------------------------------
چانک 2 | 12 کلمه | بیشتر از سقف: False
بند ۳: غیبت بیش از سه جلسه نیازمند بررسی واحد آموزش است.
----------------------------------------
چانک 3 | 25 کلمه | بیشتر از سقف: False
بند ۴: درخواست غیبت موجه باید حداکثر تا ۴۸ ساعت پس از جلسه ثبت شود.
بند ۵: تکالیف هفتگی تا ساعت ۲۳:۵۹ جمعه دریافت می‌شوند.
----------------------------------------
چانک 4 | 15 کلمه | بیشتر از سقف: False
بند ۶: دانشجو باید پیش از پایان مهلت از ثبت موفق فایل اطمینان حاصل کند.
----------------------------------------
چانک 5 | 41 کلمه | بیشتر از سقف: True
بند ۷: اگر دانشجو به دلیل اختلال سامانه نتواند تکلیف را در مهلت مقرر ارسال کند، موظف است حداکثر تا ۲۴ ساعت پس از پایان مهلت تصویر خطای سامانه و مشخصات تکلیف را به پشتیبانی بفرستد تا درخواست او بررسی شود.
----------------------------------------
چانک 6 | 16 کلمه | بیشتر از سقف: False
